<a href="https://colab.research.google.com/github/amritaa72/Runs-and-Wickets/blob/main/WBBL_HTML_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
url= "https://cricketsix.com/match/162733/sydney-thunder-women-vs-brisbane-heat-women-6th-match-wbbl-2026/"
response= requests.get(url, timeout=20)
print("status code :",response.status_code)
soup= BeautifulSoup(response.text, "html.parser")
print("Page Title : ",soup.title.get_text(strip=True) if soup.title else "Not Found")

page_text = soup.get_text("\n", strip=True)

keyword = "Ball-by-ball commentary"
position = page_text.find(keyword)

if position != -1:
    print("\nCommentary heading found!")
    print(page_text[position:position + 2500])
else:
    print("\nCommentary heading NOT found in downloaded HTML.")


status code : 200
Page Title :  Sydney Thunder Women vs Brisbane Heat Women Scorecard & Commentary, 6th Match | WBBL 2026 | CricketSix.com

Commentary heading found!
Ball-by-ball commentary
All
Wickets
4s & 6s
Overs
16.4
4
Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!
Nadine de Klerk comes in, a right-handed bat.
16.3
W
Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught by Grace Harris off Lily Bassingthwaighte, a well-made 35 from just 16 balls, with 6 fours and a six, Harris departs! Laura Harris c Grace Harris b Lily Bassingthwaighte 35(16) [4s-6 6s-1]
Wicket, caught.
16.3
Lily Bassingthwaighte to Laura Harris, wide.
16.2
4
Lily Bassingthwaighte to Laura Harris, FOUR, Laura Harris pulls it away to deep square leg, and it races across the rope for four!
16.1
Lily Bassingthwaighte to Heather Kn

# **inspect the HTML structure**

In [36]:

# Inspect the HTML structure around the commentary section

heading = soup.find(
    string=lambda text: text and "Ball-by-ball commentary" in text
)

if heading:
    element = heading.parent

    print("COMMENTARY HEADING ANCESTORS\n")

    for level in range(6):
        if element is None:
            break

        print(f"\nLEVEL {level}")
        print("Tag:", element.name)
        print("Attributes:", element.attrs)
        print("Text:", element.get_text(" ", strip=True)[:500])

        element = element.parent


# Inspect HTML elements containing a delivery number

print("\n\nDELIVERY ELEMENTS\n")

for node in soup.find_all(string=lambda text: text and text.strip() == "16.4")[:5]:
    element = node.parent

    print("\nDelivery candidate:")
    print(element.prettify()[:1500])


COMMENTARY HEADING ANCESTORS


LEVEL 0
Tag: h2
Attributes: {'id': 'h-comm'}
Text: Ball-by-ball commentary

LEVEL 1
Tag: section
Attributes: {'class': ['comm-sec'], 'aria-labelledby': 'h-comm'}
Text: Ball-by-ball commentary All Wickets 4s & 6s Overs 16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four! Nadine de Klerk comes in, a right-handed bat. 16.3 W Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught by Grace Harris off Lily Bassingthwaighte, a well-made 35 from just 16 balls, with 6 fours and a si

LEVEL 2
Tag: div
Attributes: {'class': ['m-main']}
Text: BRHW 178/5  ·  SYTW 180/4 Sydney Thunder Women won by 6 wkts Read the match report Sydney Thunder Women beat Brisbane Heat Women by 6 wickets Highlights Official videos, played by YouTube. Watching one means accepting the YouTube Terms of Ser

# **Find the individual delivery containers**

In [37]:

commentary = soup.select_one("section.comm-sec")

balls = commentary.select("span.ball-over")

print("Total ball-number elements:", len(balls))

for i, ball in enumerate(balls[:5]):
    print(f"\n{'='*60}")
    print("DELIVERY", i + 1)

    element = ball

    for level in range(5):
        if element is None:
            break

        print(
            f"Level {level}:",
            element.name,
            element.get("class"),
            "|",
            element.get_text(" ", strip=True)[:350]
        )

        element = element.parent


Total ball-number elements: 178

DELIVERY 1
Level 0: span ['ball-over'] | 16.4 4
Level 1: li ['ball', 'ball-b'] | 16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!
Level 2: ol ['comm'] | 16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four! Nadine de Klerk comes in, a right-handed bat. 16.3 W Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught
Level 3: section ['comm-sec'] | Ball-by-ball commentary All Wickets 4s & 6s Overs 16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four! Nadine de Klerk comes in, a right-handed bat. 16.3 W Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harr

## **DIAGNOSTIC CELL**

# **Extract all delivery records**

In [38]:
commentary = soup.select_one("section.comm-sec")
delivery_elements = commentary.select("li.ball:not(.ball-note)")

records = []

for delivery in delivery_elements:

    ball_element = delivery.select_one("span.ball-over")

    # Extract only the direct text node: 16.4
    ball_number = ball_element.contents[0].strip()

    # Extract indicator: 4, 6, W, etc.
    indicator_element = ball_element.select_one(".bb")
    indicator = indicator_element.get_text(strip=True) if indicator_element else None

    # Extract commentary paragraph only
    description = delivery.select_one(".ball-text p").get_text(" ", strip=True)

    records.append({
        "ball_number": ball_number,
        "indicator": indicator,
        "html_class": " ".join(delivery.get("class", [])),
        "commentary": description
    })

df = pd.DataFrame(records)

# Inspect the two deliveries from Lucy Hamilton at 14.3

for i, record in enumerate(records):
    if "Lucy Hamilton to Laura Harris, FOUR" in record["commentary"]:
        print("Record number:", i)
        print("Ball number in records:", repr(record["ball_number"]))

        print("Ball number in dataframe:",
              repr(df.loc[i, "ball_number"]))

        print("-------------------------")


print("CHECKING RECORDS BEFORE DATAFRAME")

for record in records:
    if "Lucy Hamilton" in record["commentary"]:
        print(record["ball_number"])
        print(record["commentary"])

print("Total deliveries:", len(df))

display(df.head(10))

Record number: 14
Ball number in records: '14.3'
Ball number in dataframe: '14.3'
-------------------------
CHECKING RECORDS BEFORE DATAFRAME
14.6
Lucy Hamilton to Laura Harris, 1 run, Laura Harris works it away to deep mid-wicket for a single.
14.5
Lucy Hamilton to Heather Knight, 1 run, Heather Knight clips it out towards deep mid-wicket for a single.
14.4
Lucy Hamilton to Laura Harris, 1 run, Laura Harris pushes it out towards sweeper cover for a single.
14.3
Lucy Hamilton to Laura Harris, FOUR, Laura Harris sweeps it away to backward square leg for four!
14.2
Lucy Hamilton to Laura Harris, SIX, Laura Harris connects cleanly and launches it all the way over deep mid-wicket for six!
14.1
Lucy Hamilton to Heather Knight, 3 runs, Heather Knight pings this into the deep backward point region, and they'll easily scamper back for three good runs!
12.6
Lucy Hamilton to Laura Harris, no run, A dot ball from Lucy Hamilton to Laura Harris, good defence there.
12.5
Lucy Hamilton to Laura Harri

,ball_number,indicator,html_class,commentary
0,16.4,4,ball ball-b,"Lily Bassingthwaighte to Nadine de Klerk, FOUR..."
1,16.3,W,ball ball-w,"Lily Bassingthwaighte to Laura Harris, out Cau..."
2,16.3,None,ball,"Lily Bassingthwaighte to Laura Harris, wide."
3,16.2,4,ball ball-b,"Lily Bassingthwaighte to Laura Harris, FOUR, L..."
4,16.1,None,ball,"Lily Bassingthwaighte to Heather Knight, 1 run..."
5,15.6,None,ball,"Sarah Glenn to Heather Knight, 1 run, Heather ..."
6,15.5,4,ball ball-b,"Sarah Glenn to Heather Knight, FOUR, Knight cl..."
7,15.4,4,ball ball-b,"Sarah Glenn to Heather Knight, FOUR, Knight dr..."
8,15.3,None,ball,"Sarah Glenn to Laura Harris, 1 run, Laura Harr..."
9,15.2,None,ball,"Sarah Glenn to Laura Harris, no run."


In [39]:
"""
print("Original dataframe:")
print(df.loc[14, ["ball_number", "commentary"]].to_dict())

print("\nParsed dataframe:")
print(parsed.loc[14, ["ball_number", "commentary"]].to_dict())

print("\nAll 14.3 entries in original dataframe:")
print(
    df[df["ball_number"] == "14.3"][
        ["ball_number", "commentary"]
    ].to_string(index=False)
)

print("\nAll 14.3 entries in parsed dataframe:")
print(
    parsed[parsed["ball_number"] == "14.3"][
        ["ball_number", "commentary"]
    ].to_string(index=False)
)
"""

'\nprint("Original dataframe:")\nprint(df.loc[14, ["ball_number", "commentary"]].to_dict())\n\nprint("\nParsed dataframe:")\nprint(parsed.loc[14, ["ball_number", "commentary"]].to_dict())\n\nprint("\nAll 14.3 entries in original dataframe:")\nprint(\n    df[df["ball_number"] == "14.3"][\n        ["ball_number", "commentary"]\n    ].to_string(index=False)\n)\n\nprint("\nAll 14.3 entries in parsed dataframe:")\nprint(\n    parsed[parsed["ball_number"] == "14.3"][\n        ["ball_number", "commentary"]\n    ].to_string(index=False)\n)\n'

# **Separate deliveries from commentary notes**

In [40]:

# Keep only actual delivery entries
delivery_records = []

for delivery in delivery_elements:

    classes = delivery.get("class", [])

    # Exclude player introductions, wicket notes and over updates
    if "ball-note" in classes:
        continue

    ball_element = delivery.select_one("span.ball-over")
    run_element = ball_element.select_one(".bb") if ball_element else None

    ball_number = ball_element.contents[0].strip() if ball_element else None
    indicator = run_element.get_text(strip=True) if run_element else None

    delivery_records.append({
        "ball_number": ball_number,
        "indicator": indicator,
        "html_class": " ".join(classes),
        "commentary": delivery.get_text(" ", strip=True)
    })

df = pd.DataFrame(delivery_records)

print("Total HTML elements:", len(delivery_elements))
print("Actual delivery records:", len(df))
print("\nClass distribution:")
print(df["html_class"].value_counts())

display(df.head(10))


Total HTML elements: 178
Actual delivery records: 178

Class distribution:
html_class
ball                  127
ball ball-b            33
ball ball-b ball-6     10
ball ball-w             8
Name: count, dtype: int64


,ball_number,indicator,html_class,commentary
0,16.4,4,ball ball-b,16.4 4 Lily Bassingthwaighte to Nadine de Kler...
1,16.3,W,ball ball-w,"16.3 W Lily Bassingthwaighte to Laura Harris, ..."
2,16.3,None,ball,"16.3 Lily Bassingthwaighte to Laura Harris, wide."
3,16.2,4,ball ball-b,"16.2 4 Lily Bassingthwaighte to Laura Harris, ..."
4,16.1,None,ball,"16.1 Lily Bassingthwaighte to Heather Knight, ..."
5,15.6,None,ball,"15.6 Sarah Glenn to Heather Knight, 1 run, Hea..."
6,15.5,4,ball ball-b,"15.5 4 Sarah Glenn to Heather Knight, FOUR, Kn..."
7,15.4,4,ball ball-b,"15.4 4 Sarah Glenn to Heather Knight, FOUR, Kn..."
8,15.3,None,ball,"15.3 Sarah Glenn to Laura Harris, 1 run, Laura..."
9,15.2,None,ball,"15.2 Sarah Glenn to Laura Harris, no run."


In [41]:
print("Records:")
print(repr(records[14]["ball_number"]))

print("\nDataFrame:")
print(repr(df.iloc[14]["ball_number"]))

print("\nDataFrame created from records again:")
test_df = pd.DataFrame(records)
print(repr(test_df.iloc[14]["ball_number"]))

print("\nAre the values different?")
print(df.iloc[14]["ball_number"] != test_df.iloc[14]["ball_number"])

Records:
'14.3'

DataFrame:
'14.3'

DataFrame created from records again:
'14.3'

Are the values different?
False


# **Extract batter, bowler and delivery outcome**

In [42]:
import re

def parse_delivery(row):

    text = row["commentary"]
    html_class = row["html_class"]

    # Remove ball number and indicator from commentary
    clean_text = re.sub(
        r"^\d+\.\d+\s*(?:[46W]\s*)?",
        "",
        text
    ).strip()

    # Extract bowler and batter
    match = re.match(r"(.+?) to (.+?),", clean_text)

    bowler = match.group(1).strip() if match else None
    batter = match.group(2).strip() if match else None

    lower = clean_text.lower()

    # Identify outcome
    if "ball-w" in html_class:
        outcome = "wicket"
    elif "ball-6" in html_class:
        outcome = "six"
    elif "ball-b" in html_class:
        outcome = "four"
    elif "wide" in lower:
        outcome = "wide"
    elif "no ball" in lower or "no-ball" in lower:
        outcome = "no_ball"
    elif "no run" in lower:
        outcome = "dot"
    elif "leg bye" in lower:
        outcome = "leg_bye"
    elif re.search(r"\bbye\b", lower):
        outcome = "bye"
    else:
        outcome = "runs"

    # Identify dismissal
    dismissal = None

    if outcome == "wicket":
        for dtype in [
            "run out", "caught", "bowled", "lbw",
            "stumped", "hit wicket", "retired hurt",
            "retired out", "obstructing the field"
        ]:
            if dtype in lower:
                dismissal = dtype
                break

    # Extract runs
    runs_match = re.search(r"\b(\d+)\s+runs?\b", lower)

    if outcome == "six":
        runs = 6
    elif outcome == "four":
        runs = 4
    elif outcome == "dot":
        runs = 0
    elif runs_match:
        runs = int(runs_match.group(1))
    else:
        runs = None

    return pd.Series({
        "bowler": bowler,
        "batter": batter,
        "outcome": outcome,
        "runs_from_commentary": runs,
        "dismissal_type": dismissal
    })


# Rebuild parsed dataframe
parsed = df.join(df.apply(parse_delivery, axis=1))

display(parsed.head(20))

,ball_number,indicator,html_class,commentary,bowler,batter,outcome,runs_from_commentary,dismissal_type
0,16.4,4,ball ball-b,16.4 4 Lily Bassingthwaighte to Nadine de Kler...,Lily Bassingthwaighte,Nadine de Klerk,four,4.0,None
1,16.3,W,ball ball-w,"16.3 W Lily Bassingthwaighte to Laura Harris, ...",Lily Bassingthwaighte,Laura Harris,wicket,NaN,caught
2,16.3,None,ball,"16.3 Lily Bassingthwaighte to Laura Harris, wide.",Lily Bassingthwaighte,Laura Harris,wide,NaN,None
3,16.2,4,ball ball-b,"16.2 4 Lily Bassingthwaighte to Laura Harris, ...",Lily Bassingthwaighte,Laura Harris,four,4.0,None
4,16.1,None,ball,"16.1 Lily Bassingthwaighte to Heather Knight, ...",Lily Bassingthwaighte,Heather Knight,runs,1.0,None
5,15.6,None,ball,"15.6 Sarah Glenn to Heather Knight, 1 run, Hea...",Sarah Glenn,Heather Knight,runs,1.0,None
6,15.5,4,ball ball-b,"15.5 4 Sarah Glenn to Heather Knight, FOUR, Kn...",Sarah Glenn,Heather Knight,four,4.0,None
7,15.4,4,ball ball-b,"15.4 4 Sarah Glenn to Heather Knight, FOUR, Kn...",Sarah Glenn,Heather Knight,four,4.0,None
8,15.3,None,ball,"15.3 Sarah Glenn to Laura Harris, 1 run, Laura...",Sarah Glenn,Laura Harris,runs,1.0,None
9,15.2,None,ball,"15.2 Sarah Glenn to Laura Harris, no run.",Sarah Glenn,Laura Harris,dot,0.0,None


## **DIAGNOSTIC**

In [43]:
print("Values in records:")

print([
    r["ball_number"]
    for r in records
    if "14.3" in r["commentary"]
])

print("\nValues in df:")

print(
    df.loc[
        df["commentary"].str.contains("14.3", na=False),
        "ball_number"
    ].tolist()
)

Values in records:
[]

Values in df:
['14.3', '14.3']


In [44]:
print([
    record["ball_number"]
    for record in records
    if "14.3" in record["commentary"]
])

[]


In [45]:
print(
    df.loc[
        df["commentary"].str.contains("14.3", na=False),
        "ball_number"
    ].tolist()
)

print("Data type:", df["ball_number"].dtype)

['14.3', '14.3']
Data type: object


# **Inspect one raw delivery**

In [46]:
commentary = soup.select_one("section.comm-sec")

for delivery in commentary.select("li.ball"):
    if delivery.select_one("span.ball-over"):
        print(delivery)
        break

<li class="ball ball-b" data-ts="1793518334461"><span class="ball-over">16.4<span class="bb bb-b bb-4" title="Four">4</span></span><div class="ball-text"><p>Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!</p></div></li>


In [47]:
print(df.columns.tolist())
print(parsed.columns.tolist())

['ball_number', 'indicator', 'html_class', 'commentary']
['ball_number', 'indicator', 'html_class', 'commentary', 'bowler', 'batter', 'outcome', 'runs_from_commentary', 'dismissal_type']


In [48]:
print("Commentary:", repr(df.loc[0, "commentary"]))
print("HTML class:", repr(df.loc[0, "html_class"]))

Commentary: "16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!"
HTML class: 'ball ball-b'


# **DIAGNOSTIC CELL**

In [49]:
# Find the delivery whose ball number is 14.3
for delivery in delivery_elements:

    ball_element = delivery.select_one("span.ball-over")

    if ball_element and "14.3" in ball_element.get_text():

        print("HTML:")
        print(ball_element)

        print("\nDirect text node:")
        print(repr(ball_element.contents[0]))

        print("\nAll text nodes:")
        print(ball_element.find_all(string=True))

        print("\nExtracted using contents[0]:")
        print(repr(ball_element.contents[0].strip()))

        break

HTML:
<span class="ball-over">14.3<span class="bb bb-b bb-4" title="Four">4</span></span>

Direct text node:
'14.3'

All text nodes:
['14.3', '4']

Extracted using contents[0]:
'14.3'


In [50]:
print(
    df.loc[
        df["commentary"].str.contains("14.3", na=False),
        ["ball_number", "indicator", "commentary"]
    ].to_string(index=False)
)

ball_number indicator                                                                                                            commentary
       14.3         4              14.3 4 Lucy Hamilton to Laura Harris, FOUR, Laura Harris sweeps it away to backward square leg for four!
       14.3         4 14.3 4 Nadine de Klerk to Orla Prendergast, FOUR, Orla Prendergast gets this away fine down to the boundary for four!


In [51]:
print(
    df[df["ball_number"] == "14.3"][
        ["ball_number", "indicator", "commentary"]
    ].to_string(index=False)
)

print("\nTotal deliveries:", len(df))

ball_number indicator                                                                                                            commentary
       14.3         4              14.3 4 Lucy Hamilton to Laura Harris, FOUR, Laura Harris sweeps it away to backward square leg for four!
       14.3         4 14.3 4 Nadine de Klerk to Orla Prendergast, FOUR, Orla Prendergast gets this away fine down to the boundary for four!

Total deliveries: 178


## **validate the extraction**

In [52]:
print("Duplicate ball numbers:")
print(df[df.duplicated("ball_number", keep=False)][
    ["ball_number", "indicator", "html_class", "commentary"]
].to_string(index=False))

print("\nIndicator distribution:")
print(df["indicator"].value_counts(dropna=False))

print("\nDelivery count:", len(df))

Duplicate ball numbers:
ball_number indicator         html_class                                                                                                                                                                                                                                                                                                                                       commentary
       16.4         4        ball ball-b                                                                                                                                                                                     16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!
       16.3         W        ball ball-w 16.3 W Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught by Grace Harris off Lily Bassin

## **identify the innings**

In [53]:
print("Commentary section details:")

for section in soup.select("section.comm-sec"):
    print(
        "Class:", section.get("class"),
        "| Heading:", section.find_previous(["h2", "h3"]).get_text(strip=True)
        if section.find_previous(["h2", "h3"]) else "No heading",
        "| Deliveries:", len(section.select("li.ball:not(.ball-note)"))
    )

Commentary section details:
Class: ['comm-sec'] | Heading: Highlights | Deliveries: 178


# **looking for labels such as 1st Innings, 2nd Innings, team names, or innings breaks.**

In [54]:
notes = commentary.select("li.ball-note")

print("Number of notes:", len(notes))
print("\nNotes found:\n")

for note in notes:
    print(note.get_text(" ", strip=True))

Number of notes: 40

Notes found:

Nadine de Klerk comes in, a right-handed bat.
Wicket, caught.
Lily Bassingthwaighte comes back, 3.0-0-21-1 so far.
Sarah Glenn comes back, 2.0-0-26-0 so far.
Laura Harris comes in, a right-handed bat.
Wicket, caught.
Lucy Hamilton comes back, 1.0-0-9-0 so far.
Sianna Ginger comes back, 1.0-0-6-0 so far.
Orla Prendergast comes back, 1.0-0-8-1 so far.
Drinks break.
Sianna Ginger comes on, right-arm fast-medium.
Charli Knott comes on, right-arm off-break.
Heather Knight comes in, a right-handed bat.
Wicket, caught.
Sarah Glenn comes on, right-arm leg-break.
Anika Learoyd comes in, a right-handed bat.
Wicket, caught.
Orla Prendergast comes on, right-arm fast-medium.
Lily Bassingthwaighte comes on, right-arm medium.
Phoebe Litchfield and Georgia Voll open the batting, Phoebe Litchfield on strike. Lucy Hamilton opens the attack.
Lucy Hamilton comes in, a left-handed bat.
Wicket, stumped.
Nadine de Klerk comes back, 3.0-0-24-0 so far.
Annie O Neil comes in, 

## **found : they are event descriptions, not innings labels**

# **inspect the HTML around the innings transitions**

In [55]:
for i, item in enumerate(commentary.select("li")):
    text = item.get_text(" ", strip=True)

    if "open the batting" in text.lower():
        print(f"\nPossible innings start at HTML item {i}:")
        print("NOTE:", text)

        print("\nNext 5 deliveries:")
        count = 0

        for following in commentary.select("li")[i+1:]:
            if "ball-note" not in following.get("class", []):
                print(following.get_text(" ", strip=True))
                count += 1

            if count == 5:
                break


Possible innings start at HTML item 142:
NOTE: Phoebe Litchfield and Georgia Voll open the batting, Phoebe Litchfield on strike. Lucy Hamilton opens the attack.

Next 5 deliveries:
End of over 20 10 runs · 178-5 4 0 3 1 W 2 4 0 3 1 W 2 Lucy Hamilton 2(1) · Annie O Neil 8(4) · Nadine de Klerk 4-0-34-1
19.6 Nadine de Klerk to Lucy Hamilton, 2 runs, Lucy Hamilton works it out towards long-on, easily coming back for a couple of runs!
19.5 W Nadine de Klerk to Orla Prendergast, out Stumped!! She is drawn right out of her crease and beaten all ends up, and Tahlia Wilson whips the bails off in a flash! Orla Prendergast has to go, stumped by Tahlia Wilson off Nadine de Klerk! Orla Prendergast st Tahlia Wilson b Nadine de Klerk 28(20) [4s-5]
19.4 Nadine de Klerk to Annie O Neil, 1 run, Annie O Neil works it away to long-off for a single.
19.3 Nadine de Klerk to Orla Prendergast, 3 runs, Orla Prendergast nudges it away down to fine leg and they'll easily come back for three!


-  **note's position in the HTML is not chronological**
- **need to stop using HTML item order to assign innings.**
- **inspecting delivery timestamps**

In [56]:
for item in commentary.select("li.ball:not(.ball-note)")[:10]:
    print(
        "Ball:", item.select_one(".ball-over").contents[0].strip(),
        "| Timestamp:", item.get("data-ts"),
        "| Commentary:", item.select_one(".ball-text p").get_text(" ", strip=True)
    )

Ball: 16.4 | Timestamp: 1793518334461 | Commentary: Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!
Ball: 16.3 | Timestamp: 1793518234543 | Commentary: Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught by Grace Harris off Lily Bassingthwaighte, a well-made 35 from just 16 balls, with 6 fours and a six, Harris departs! Laura Harris c Grace Harris b Lily Bassingthwaighte 35(16) [4s-6 6s-1]
Ball: 16.3 | Timestamp: 1793518200058 | Commentary: Lily Bassingthwaighte to Laura Harris, wide.
Ball: 16.2 | Timestamp: 1793518151580 | Commentary: Lily Bassingthwaighte to Laura Harris, FOUR, Laura Harris pulls it away to deep square leg, and it races across the rope for four!
Ball: 16.1 | Timestamp: 1793518124747 | Commentary: Lily Bassingthwaighte to Heather Knight, 1 run, Heather Knight works this out to dee

In [57]:
for item in commentary.select("li.ball:not(.ball-note)")[-10:]:
    print(
        "Ball:", item.select_one(".ball-over").contents[0].strip(),
        "| Timestamp:", item.get("data-ts"),
        "| Commentary:", item.select_one(".ball-text p").get_text(" ", strip=True)
    )

Ball: 10.2 | Timestamp: 1793510518437 | Commentary: Samantha Bates to Redmayne, 2 runs, Georgia Redmayne works this out to deep mid-wicket and they sprint hard for a couple of runs!
Ball: 10.1 | Timestamp: 1793510480771 | Commentary: Samantha Bates to Grace Harris, 1 run, Grace Harris works it away towards long-on for a single.
Ball: 9.6 | Timestamp: 1793510280143 | Commentary: Jemma Barsby to Grace Harris, 1 run, Grace Harris pushes this out toward deep mid-wicket for a single.
Ball: 9.5 | Timestamp: 1793510246653 | Commentary: Jemma Barsby to Grace Harris, SIX, Grace Harris connects cleanly, launching it high and far over long-on for a maximum!
Ball: 9.4 | Timestamp: 1793510219294 | Commentary: Jemma Barsby to Grace Harris, 2 runs, Grace Harris clips it away towards backward square leg, and they'll comfortably hustle back for the second.
Ball: 9.3 | Timestamp: 1793510195734 | Commentary: Jemma Barsby to Redmayne, 1 run, Georgia Redmayne works this out to deep backward point for a sin

- **We need to sort by timestamp.**
- **Check the timestamp range**

In [58]:
timestamp_data = []

for item in commentary.select("li.ball:not(.ball-note)"):
    timestamp_data.append({
        "timestamp": int(item["data-ts"]),
        "ball_number": item.select_one(".ball-over").contents[0].strip(),
        "commentary": item.select_one(".ball-text p").get_text(" ", strip=True)
    })

time_df = pd.DataFrame(timestamp_data)

time_df = time_df.sort_values("timestamp").reset_index(drop=True)

print("First 10 chronological deliveries:")
print(time_df.head(10)[["timestamp", "ball_number", "commentary"]].to_string(index=False))

print("\nLast 10 chronological deliveries:")
print(time_df.tail(10)[["timestamp", "ball_number", "commentary"]].to_string(index=False))

First 10 chronological deliveries:
    timestamp ball_number                                                                                                                                        commentary
1793510057330         8.5                                                                  Nadine de Klerk to Redmayne, FOUR, Redmayne drives it away to long-off for four!
1793510069979         8.6                                                Nadine de Klerk to Redmayne, no run, Nadine de Klerk keeps it quiet to Georgia Redmayne, dot ball.
1793510155990         9.1                               Jemma Barsby to Grace Harris, 1 run, Grace Harris works this out to deep extra cover and jogs through for a single.
1793510157175         9.2                                          Jemma Barsby to Redmayne, no run, Jemma Barsby gets it through to Georgia Redmayne, and there is no run.
1793510195734         9.3                                             Jemma Barsby to Redmayne, 1 run, Ge

# **inspect the full timestamp range and missing overs**

In [59]:
time_df

,timestamp,ball_number,commentary
0,1793510057330,8.5,"Nadine de Klerk to Redmayne, FOUR, Redmayne dr..."
1,1793510069979,8.6,"Nadine de Klerk to Redmayne, no run, Nadine de..."
2,1793510155990,9.1,"Jemma Barsby to Grace Harris, 1 run, Grace Har..."
3,1793510157175,9.2,"Jemma Barsby to Redmayne, no run, Jemma Barsby..."
4,1793510195734,9.3,"Jemma Barsby to Redmayne, 1 run, Georgia Redma..."
...,...,...,...
173,1793518124747,16.1,"Lily Bassingthwaighte to Heather Knight, 1 run..."
174,1793518151580,16.2,"Lily Bassingthwaighte to Laura Harris, FOUR, L..."
175,1793518200058,16.3,"Lily Bassingthwaighte to Laura Harris, wide."
176,1793518234543,16.3,"Lily Bassingthwaighte to Laura Harris, out Cau..."


In [60]:
print("Earliest timestamp : ",time_df["timestamp"].min())
print('Latest timestamp : ',time_df["timestamp"].max())

print("\n Delivery count by over :")

time_df["over"]=time_df["ball_number"].str.split(".").str[0].astype(int)

print(time_df.groupby("over").size().to_string())

print("\nFirst and last ball numbers : ")
print(time_df["ball_number"].iloc[0])
print(time_df["ball_number"].iloc[-1])



Earliest timestamp :  1793510057330
Latest timestamp :  1793518334461

 Delivery count by over :
over
0      8
1      6
2      6
3      6
4      6
5      7
6      6
7      7
8      8
9     12
10    12
11    14
12    13
13    14
14    12
15    12
16    11
17     6
18     6
19     6

First and last ball numbers : 
8.5
16.4


In [61]:
all_deliveries = commentary.select("li.ball:not(.ball-note)")

for i in range(85, 100):
    item = all_deliveries[i]

    print(
        i,
        "|",
        item.select_one(".ball-over").contents[0].strip(),
        "|",
        item.select_one(".ball-text p").get_text(" ", strip=True)[:110]
    )

85 | 3.2 | Lily Bassingthwaighte to Anika Learoyd, no run, And Lily Bassingthwaighte keeps it tight to Anika Learoyd, not
86 | 3.1 | Lily Bassingthwaighte to Anika Learoyd, no run, Anika Learoyd defends it solidly back, no run taken.
87 | 2.6 | Orla Prendergast to Anika Learoyd, 1 run, Learoyd taps it away toward backward point and scampers through for 
88 | 2.5 | Orla Prendergast to Georgia Voll, out Caught by Redmayne!! What a breakthrough, Prendergast strikes, Voll caug
89 | 2.4 | Orla Prendergast to Georgia Voll, no run, Dot ball, Orla Prendergast keeps it tight to Georgia Voll.
90 | 2.3 | Orla Prendergast to Phoebe Litchfield, 1 run, Phoebe Litchfield works it away to square leg for a single.
91 | 2.2 | Orla Prendergast to Phoebe Litchfield, FOUR, Phoebe Litchfield threads that away to the deep mid-wicket bounda
92 | 2.1 | Orla Prendergast to Phoebe Litchfield, 2 runs, Phoebe Litchfield works it away towards backward square leg, an
93 | 1.6 | Lily Bassingthwaighte to Georgia Voll,

## **locate the exact transition**

In [62]:
all_deliveries = commentary.select("li.ball:not(.ball-note)")

for i in range(65, 90):
    item = all_deliveries[i]

    print(
        i,
        "|",
        item.select_one(".ball-over").contents[0].strip(),
        "|",
        item.select_one(".ball-text p").get_text(" ", strip=True)
    )

65 | 6.3 | Sarah Glenn to Heather Knight, 1 run, Heather Knight works it out towards long-off for a single.
66 | 6.2 | Sarah Glenn to Anika Learoyd, 1 run, Worked away out to deep mid-wicket by Anika Learoyd for a single.
67 | 6.1 | Sarah Glenn to Anika Learoyd, no run, Sarah Glenn pushes one through to Anika Learoyd, defended back and no run.
68 | 5.6 | Lily Bassingthwaighte to Phoebe Litchfield, out Caught by Orla Prendergast!! Phoebe Litchfield has to go! Orla Prendergast takes the catch off Lily Bassingthwaighte, and that is the end of a sparkling 41! Phoebe Litchfield c Orla Prendergast b Lily Bassingthwaighte 41(19) [4s-6 6s-1]
69 | 5.5 | Lily Bassingthwaighte to Phoebe Litchfield, FOUR, Phoebe Litchfield drives it away towards long-off and races all the way for four!
70 | 5.5 | Lily Bassingthwaighte to Phoebe Litchfield, wide, Wayward from Lily Bassingthwaighte, it's called a wide to Phoebe Litchfield.
71 | 5.4 | Lily Bassingthwaighte to Phoebe Litchfield, SIX, Phoebe Litchfield